In [1]:
import os
%pwd

'c:\\Users\\Aswin\\MLprojects\\Quality_Of_wine(EndtoEnd ML)\\Quality_Of_Wine-EndtoEnd-\\research'

In [ ]:
os.chdir("..")

## config entity(THE DATA MODEL)- entity


In [ ]:
# config entity(THE DATA MODEL) 

""" 
The Config Entity represents the schema of your inputs. It is usually a namedtuple or a dataclass.
PURPOSE: It defines exactly what information (file paths, URLs, directories) the ingestion process
needs to run. It ensures that throughout the code, you have a consistent, read-only object to 
reference rather than messy hard-coded strings.


from the config.yaml(where the data ingestion artifact file is resent) 
the path is directed 

this is an ENTITY, a return type for a function

(THE DATA MODEL)

""" 

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

## Configuration Manager(THE CENTRAL CONTROLLER)

In [ ]:
# Configuration Manager's packages to import for the process

from Quality_of_Wine.constants import *
from Quality_of_Wine.utils.common import read_yaml,create_directories

In [ ]:
# Configuration Manager(THE CENTRAL CONTROLLER)

"""
The Configuration Manager represents the bridge between your settings and your code.
PURPOSE: It reads your external configuration files (like config.yaml or params.yaml)
and your environment variables. It is responsible for initializing the folders (artifacts)
on your system and "packaging" the specific settings into the Config Entity.

(THE CENTRAL CONTROLLER)

"""
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    """
    
    This function, usually found inside the Configuration Manager, represents a specialized request.
    PURPOSE: It goes into the large global configuration file, pulls out only the pieces relevant to
    ingestion (like the source URL and the download directory), and returns them as a Config Entity.
    It keeps the ingestion logic isolated from the rest of the project.

    """
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

## Data Ingestion(THE EXECUTOR)- components

In [ ]:
#Data Ingestion, for it the needed packages for the process

import os
import urllib.request as request
import zipfile
from Quality_of_Wine import logger

from Quality_of_Wine.utils.common import get_size
#this is specifically used to know the file size

In [ ]:
# Data Ingestion(THE EXECUTOR)

"""

This class represents the functional unit of work. It is the "worker" that actually performs the task.
PURPOSE: It contains the "how-to" logic. It takes the configuration it was given and executes the physical actions:
         ~download_file(): Reaching out to the internet/database.
         ~extract_zip_file(): Unpacking the data into your project's folders.

(THE EXECUTOR)

"""

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
  

## Data Ingestion Pipeline(for data ingestion)

In [ ]:
#pipeline(for data ingestion)

"""
this is the pipeline to execute the fetching of the data and unzipping it
and as well as saving it

"""


try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-01 22:27:52,970: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-01 22:27:52,981: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-01 22:27:52,983: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-01 22:27:52,987: INFO: common: created directory at: artifacts]
[2026-03-01 22:27:52,989: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-01 22:27:55,457: INFO: 2366278714: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: A7E4:321A57:178E7F:4150E8:69A4700F
Accept-Ranges: bytes
Date: Sun, 01

## modular coding for Data Ingestion in src folder

In [ ]:
"""
~Succcessfully ran the DATA INGESTION part of the project in reseach folder.

~End To End MLOPS Data Science Project Implementation With Deployment(YT Channel)=https://youtu.be/pxk1Fr33-L4?si=YXioCQhqeFOWQ_az

~Now it has to be covered into modular coding.
   the process and where it has to copied is mentioned below

   *Step 01: entity/config_entity.py
            ~ In the entity/config_entity.py copy and paste the code from config entity(the data model)markdown above.
   
   *Step 02: src/config/configuration.py
            ~ In the src/config/configuration.py copy and paste the code from Configuration Manager(THE CENTRAL CONTROLLER) markdown
              above with its packages.
            ~ the other package must be imported from src/entity.

   *Step 03: create a new file in src/components/data_ingestion.py(new file)
            ~ In the src/components/data_ingestion.py copy and paste the code from Data Ingestion(THE EXECUTOR) markdown above with packages.
   
   *Step 04: create a new file in src/pipelines/stage_01_data_ingestion.py(new file)
            ~ Before copying the pipeline code some code have to coded first that is already in src/pipelines/stage_01_data_ingestion.ipynb highlighted.
            ~ In the src/pipelines/stage_01_data_ingestion.ipynb copy and paste the code from Data Ingestion Pipeline(for data ingestion) markdown above
              with the packages.
   
   *Step 05: To test the pipeline from the above step the execution must be done from main.py
            ~ From the code from the src/pipelines/stage_01_data_ingestion.ipynb copy and paste the code with the packages in the main.py.
            ~ Delete the artifacts with the data.zip and unzipped data.
            ~ And execute it in the terminal -python main.py.

"""